# Placement temporal holdout: V0 vs V4

이 Notebook은 기존 26게임으로만 두 모델을 학습하고, 그 뒤 수집된 game을 temporal holdout으로 평가한다. 바꾸는 것은 pair representation과 Phase 8에서 결합된 weighting 계약뿐이다.

권장 읽기 순서: **cohort 분리 → worked example → shared scaler와 V0/V4 → choice-required physical-class metric → user full-run gate**.

## 1. 질문과 고정 조건

질문: 26게임으로만 학습했을 때 이후 신규 game에서 V0와 V4 중 어느 표현이 더 안정적인가?

고정 조건은 기존 16-feature, base-only StandardScaler, LogisticRegression 설정, canonical tie-break다. V0는 raw candidate multiplicity를 유지하고 V4는 occupied-cell physical class당 대표 pair 하나와 class-balanced weight를 사용한다. Smoke는 구현 경로 확인용이며 성능 결론을 허용하지 않는다.


In [ ]:
from __future__ import annotations

import hashlib
import io
import json
import os
import platform
import tempfile
import warnings
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import scipy
import sklearn
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

EXPERIMENT_ID = "placement-temporal-holdout-v0-v4"
VARIANT_NAMES = ["V0", "V4"]
VARIANT_DEFINITIONS = {
    "V0": "raw selected-vs-all-unselected candidate pairs",
    "V4": "one representative per non-selected physical class",
}
MODEL_CONFIG = {
    "solver": "lbfgs", "fitIntercept": False, "classWeight": None,
    "tolerance": 1e-12, "maxIterations": 10000, "l2": 0.01,
}
display(pd.DataFrame([{"experimentId": EXPERIMENT_ID, **MODEL_CONFIG}]))


## 2. 경로와 source checksum

**왜 실행하는가:** live source와 immutable base를 명시하고 실행 전 byte 상태를 고정한다.

**입력:** base dataset, 기존 exporter로 만든 cumulative dataset, production source log.

**계산:** 네 dataset 파일과 source log의 SHA-256을 계산한다. Report에는 절대 경로를 쓰지 않는다.

**읽는 법:** base checksum 네 개가 Phase 8 계약과 같아야 한다.

**보류 조건:** 파일 누락이나 base checksum 불일치가 있으면 학습하지 않는다.


In [ ]:
def find_repository_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "package.json").is_file() and (candidate / "python_tests").is_dir():
            return candidate
    raise AssertionError("REPOSITORY_ROOT_NOT_FOUND")


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def resolve_config_path(value: str | Path, root: Path) -> Path:
    path = Path(value)
    return path if path.is_absolute() else root / path


ROOT = find_repository_root(Path.cwd())
BASE_DATASET = resolve_config_path(os.environ.get(
    "PLACEMENT_TEMPORAL_BASE_DATASET",
    ROOT / "data/datasets/human-behavior-v2-snapshot-26g-20260717",
), ROOT)
CUMULATIVE_DATASET = resolve_config_path(os.environ.get(
    "PLACEMENT_TEMPORAL_CUMULATIVE_DATASET",
    ROOT / "data/datasets/human-behavior-v2-temporal-cumulative-75g-20260718",
), ROOT)
SOURCE_LOG = resolve_config_path(os.environ.get(
    "PLACEMENT_TEMPORAL_SOURCE_LOG",
    ROOT.parent / "Tetris_multi/data/logs/game-events.jsonl",
), ROOT)
DATASET_FILES = ["manifest.json", "train.jsonl", "validation.jsonl", "rejected.jsonl"]
EXPECTED_BASE_SHA256 = {
    "manifest.json": "51027be0ebad578f2c38bb8715aa491daa8f13208e3da551a5e4a2dd08bef088",
    "train.jsonl": "85034a66373add54324a56673cb99ed5854810e4299abcaf9c195ae8cbca631e",
    "validation.jsonl": "23b832d95a2057b7b29e7062286c58c68524fa8db00f00870e6076a0a846f58a",
    "rejected.jsonl": "06ede14119014e2b040f8a4d62e00532ca6eecae9424b93e9cc882394db8a85f",
}
for path in [SOURCE_LOG, *[BASE_DATASET / name for name in DATASET_FILES], *[CUMULATIVE_DATASET / name for name in DATASET_FILES]]:
    assert path.is_file(), f"INPUT_REQUIRED:{path.name}"
BASE_SHA256 = {name: sha256_file(BASE_DATASET / name) for name in DATASET_FILES}
CUMULATIVE_SHA256 = {name: sha256_file(CUMULATIVE_DATASET / name) for name in DATASET_FILES}
assert BASE_SHA256 == EXPECTED_BASE_SHA256, "BASE_SNAPSHOT_CHECKSUM_MISMATCH"
SOURCE_SHA256_BEFORE = sha256_file(SOURCE_LOG)
display(pd.DataFrame([{"sourceSha256": SOURCE_SHA256_BEFORE, **BASE_SHA256}]))


## 3. Base/cumulative dataset 검증

**왜 실행하는가:** exporter split은 버리고 두 JSONL을 합치되 schema·Human label·manifest count 계약을 유지한다.

**입력:** 각 dataset의 `train.jsonl`, `validation.jsonl`, `manifest.json`.

**계산:** UTF-8/JSON, schema v2, Human source, boolean label, 정확히 하나의 selected candidate를 검증한다.

**읽는 법:** original exporter split은 provenance일 뿐 temporal cohort가 아니다.

**보류 조건:** manifest count나 decision label이 실제 row와 다르면 중단한다.


In [ ]:
def read_json_object(path: Path) -> dict[str, Any]:
    value = json.loads(path.read_bytes().decode("utf-8", errors="strict"))
    assert isinstance(value, dict), f"JSON_OBJECT_REQUIRED:{path.name}"
    return value


def read_jsonl_frame(path: Path) -> pd.DataFrame:
    text = path.read_bytes().decode("utf-8", errors="strict")
    lines = text.splitlines()
    assert lines and all(line.strip() for line in lines), f"JSONL_LINE_REQUIRED:{path.name}"
    for line_number, line in enumerate(lines, 1):
        assert isinstance(json.loads(line), dict), f"JSON_OBJECT_LINE_REQUIRED:{path.name}:{line_number}"
    frame = pd.read_json(io.StringIO(text), lines=True, dtype=False)
    assert len(frame) == len(lines), f"ROW_COUNT_MISMATCH:{path.name}"
    return frame


def load_dataset(dataset_path: Path) -> tuple[dict[str, Any], pd.DataFrame, pd.DataFrame]:
    manifest = read_json_object(dataset_path / "manifest.json")
    train = read_jsonl_frame(dataset_path / "train.jsonl").assign(originalSplit="train")
    validation = read_jsonl_frame(dataset_path / "validation.jsonl").assign(originalSplit="validation")
    rejected = read_jsonl_frame(dataset_path / "rejected.jsonl")
    assert len(train) == manifest["trainSampleCount"]
    assert len(validation) == manifest["validationSampleCount"]
    assert len(rejected) == manifest["rejectedPlacementLogCount"]
    candidates = pd.concat([train, validation], ignore_index=True, copy=True)
    assert len(candidates) == manifest["sampleCountBySource"]["human"]
    return manifest, candidates, rejected


BASE_MANIFEST, base_candidates, base_rejected = load_dataset(BASE_DATASET)
CUMULATIVE_MANIFEST, cumulative_candidates, cumulative_rejected = load_dataset(CUMULATIVE_DATASET)
REQUIRED_COLUMNS = {"datasetSchemaVersion", "gameId", "playerId", "pieceIndex", "source", "candidatePlacement", "candidateResult", "candidateMetricsAfter", "metricDelta", "selectedBySource", "metadata"}
for name, frame in [("base", base_candidates), ("cumulative", cumulative_candidates)]:
    assert not REQUIRED_COLUMNS.difference(frame.columns), f"MISSING_COLUMNS:{name}"
    assert (frame["datasetSchemaVersion"] == 2).all(), f"SCHEMA_MISMATCH:{name}"
    assert (frame["source"] == "human").all(), f"NON_HUMAN_SOURCE:{name}"
    assert frame["selectedBySource"].map(lambda value: isinstance(value, (bool, np.bool_))).all()
assert BASE_MANIFEST["sourceFilter"] == CUMULATIVE_MANIFEST["sourceFilter"] == "human"
assert BASE_MANIFEST["selectionPolicy"] == CUMULATIVE_MANIFEST["selectionPolicy"] == "all-completed"
assert CUMULATIVE_MANIFEST["inputSha256"] == SOURCE_SHA256_BEFORE, "CUMULATIVE_SOURCE_CHECKSUM_MISMATCH"
assert CUMULATIVE_MANIFEST["jsonParseFailureCount"] == 0
display(pd.DataFrame([{"dataset": "base", "games": base_candidates.gameId.nunique(), "candidates": len(base_candidates)}, {"dataset": "cumulative", "games": cumulative_candidates.gameId.nunique(), "candidates": len(cumulative_candidates)}]))


## 4. Temporal cohort와 leakage gate

**왜 실행하는가:** 과거 exporter split 대신 수집 시점이 다른 game 집합으로 train/validation을 다시 정의한다.

**입력:** base와 cumulative의 gameId.

**계산:** `new = cumulative - base`; smoke는 Unicode code-point 정렬상 처음 1~2게임만 고른다. Base 26게임 training은 줄이지 않는다.

**읽는 법:** train/new 교집합은 0이어야 한다. validation candidate must not enter scaler fit or pair construction.

**보류 조건:** base가 cumulative의 부분집합이 아니거나 신규 game이 없으면 중단한다.


In [ ]:
def codepoint_key(value: Any) -> tuple[int, ...]:
    return tuple(ord(character) for character in str(value))


def create_decision_key(row: pd.Series) -> str:
    return f"{row['gameId']}:{row['playerId']}:{int(row['pieceIndex'])}"


def create_placement_key(row: pd.Series) -> str:
    placement = row["candidatePlacement"]
    return f"{placement['pieceType']}:{int(placement['rotation'])}:{int(placement['row'])}:{int(placement['col'])}"


for frame in [base_candidates, cumulative_candidates]:
    frame["decisionKey"] = [create_decision_key(row) for _, row in frame.iterrows()]
    frame["placementKey"] = [create_placement_key(row) for _, row in frame.iterrows()]
    decision_contract = frame.groupby("decisionKey", sort=False).agg(selectedCount=("selectedBySource", "sum"), gameCount=("gameId", "nunique"), candidateCount=("placementKey", "size"), placementCount=("placementKey", "nunique"))
    assert (decision_contract["selectedCount"] == 1).all()
    assert (decision_contract["gameCount"] == 1).all()
    assert (decision_contract["candidateCount"] >= 2).all()
    assert (decision_contract["candidateCount"] == decision_contract["placementCount"]).all()
BASE_GAME_IDS = set(base_candidates["gameId"])
CUMULATIVE_GAME_IDS = set(cumulative_candidates["gameId"])
assert len(BASE_GAME_IDS) == 26
assert BASE_GAME_IDS.issubset(CUMULATIVE_GAME_IDS), "BASE_NOT_SUBSET_OF_CUMULATIVE"
NEW_GAME_IDS = sorted(CUMULATIVE_GAME_IDS - BASE_GAME_IDS, key=codepoint_key)
assert NEW_GAME_IDS, "NEW_GAME_COHORT_REQUIRED"
requested_limit = int(os.environ.get("PLACEMENT_TEMPORAL_MAX_VALIDATION_GAMES", "2"))
assert requested_limit > 0, "POSITIVE_VALIDATION_GAME_LIMIT_REQUIRED"
VALIDATION_GAME_IDS = NEW_GAME_IDS[:requested_limit]
FULL_COHORT_COVERED = len(VALIDATION_GAME_IDS) == len(NEW_GAME_IDS)
RUN_MODE = "full" if FULL_COHORT_COVERED else "smoke"
train_candidates = base_candidates.copy(deep=True).reset_index(drop=True)
validation_candidates = cumulative_candidates.loc[cumulative_candidates["gameId"].isin(VALIDATION_GAME_IDS)].copy(deep=True).reset_index(drop=True)
assert set(train_candidates["gameId"]).isdisjoint(validation_candidates["gameId"])
GAME_ID_LEAKAGE_COUNT = len(set(train_candidates["gameId"]) & set(validation_candidates["gameId"]))
display(pd.DataFrame([{"runMode": RUN_MODE, "baseGames": len(BASE_GAME_IDS), "newGames": len(NEW_GAME_IDS), "validationGames": len(VALIDATION_GAME_IDS), "gameIdLeakageCount": GAME_ID_LEAKAGE_COUNT}]))


## 5. 기존 16-feature와 occupied-cell class

**왜 실행하는가:** Phase 8과 동일한 feature와 label-free physical placement 정의를 재사용한다.

**입력:** candidateResult, metricDelta, candidatePlacement의 absolute cells.

**계산:** 16-feature vector와 `pieceType + sorted occupied cells` equivalence key를 만든다.

**읽는 법:** `selectedBySource`는 label 식별 전용이고 `metadata.isWinner`, `heuristicBaseline.score`는 feature가 아니다.

**보류 조건:** 같은 class 내부 feature나 즉시 결과가 다르면 중단한다.


In [ ]:
FEATURE_NAMES = [
    "candidateResult.linesCleared", "candidateResult.gameOver",
    "metricDelta.aggregateHeight", "metricDelta.maxHeight",
    "metricDelta.holes", "metricDelta.bumpiness",
    *[f"metricDelta.columnHeights[{index}]" for index in range(10)],
]
FORBIDDEN_FEATURES = {"selectedBySource", "metadata.isWinner", "winnerPlayerId", "heuristicBaseline.score", "metadata.inputSequence"}


def extract_feature_vector(row: pd.Series) -> np.ndarray:
    result, delta = row["candidateResult"], row["metricDelta"]
    vector = np.asarray([int(result["linesCleared"]), int(bool(result["gameOver"])), float(delta["aggregateHeight"]), float(delta["maxHeight"]), float(delta["holes"]), float(delta["bumpiness"]), *[float(value) for value in delta["columnHeights"]]], dtype=np.float64)
    assert vector.shape == (16,) and np.isfinite(vector).all()
    return vector


def create_equivalence_key(row: pd.Series) -> str:
    placement = row["candidatePlacement"]
    cells = sorted((int(cell["row"]), int(cell["col"])) for cell in placement["cells"])
    assert len(cells) == 4 and len(set(cells)) == 4
    encoded = ";".join(f"{row_value},{col_value}" for row_value, col_value in cells)
    return f"{placement['pieceType']}|{encoded}"


assert len(FEATURE_NAMES) == 16 and set(FEATURE_NAMES).isdisjoint(FORBIDDEN_FEATURES)
for frame in [train_candidates, validation_candidates]:
    frame["candidateIndex"] = np.arange(len(frame), dtype=np.int64)
    frame["featureVector"] = [extract_feature_vector(row) for _, row in frame.iterrows()]
    frame["equivalenceKey"] = [create_equivalence_key(row) for _, row in frame.iterrows()]
    for (_, class_key), members in frame.groupby(["decisionKey", "equivalenceKey"], sort=False):
        features = np.vstack(members["featureVector"])
        assert np.array_equal(features, np.repeat(features[:1], len(features), axis=0)), f"EQUIVALENCE_FEATURE_MISMATCH:{class_key}"
        for field in ["candidateResult", "candidateMetricsAfter", "metricDelta"]:
            encoded = members[field].map(lambda value: json.dumps(value, sort_keys=True, separators=(",", ":")))
            assert encoded.nunique() == 1, f"EQUIVALENCE_RESULT_MISMATCH:{field}"
display(pd.DataFrame({"index": range(16), "feature": FEATURE_NAMES}))


## 6. Worked example: alias가 pair weight에 미치는 영향

**왜 실행하는가:** candidate generator의 alias multiplicity가 V0 weight를 반복시키고 V4가 이를 class 단위로 바꾸는 과정을 손으로 확인한다.

**입력:** base에서 selected class 또는 non-selected class multiplicity가 2 이상인 첫 decision.

**계산:** raw alternative 수, physical class 수, class별 multiplicity와 두 weight 합을 펼친다.

**읽는 법:** V0는 raw alternative마다 `1/N`, V4는 non-selected class마다 `1/K`를 준다.

**보류 조건:** 두 variant의 represented decision weight 합이 1이 아니면 모델 구현을 진행하지 않는다.


In [ ]:
def decision_multiplicity(frame: pd.DataFrame, decision_key: str) -> pd.DataFrame:
    decision = frame.loc[frame["decisionKey"] == decision_key]
    return decision.groupby("equivalenceKey", sort=False).agg(multiplicity=("placementKey", "size"), selectedClass=("selectedBySource", "any")).reset_index()


example_candidates = []
for decision_key in sorted(train_candidates["decisionKey"].unique(), key=codepoint_key):
    multiplicity = decision_multiplicity(train_candidates, decision_key)
    if int(multiplicity["multiplicity"].max()) >= 2 and len(multiplicity) > 1:
        example_candidates.append(decision_key)
WORKED_DECISION = example_candidates[0]
worked = train_candidates.loc[train_candidates["decisionKey"] == WORKED_DECISION].copy()
worked_multiplicity = decision_multiplicity(train_candidates, WORKED_DECISION)
selected_class = worked.loc[worked["selectedBySource"], "equivalenceKey"].iloc[0]
raw_alternatives = len(worked) - 1
non_selected_classes = worked_multiplicity.loc[worked_multiplicity["equivalenceKey"] != selected_class]
worked_rows = []
for _, row in worked_multiplicity.sort_values("equivalenceKey", key=lambda column: column.map(codepoint_key)).iterrows():
    if row["equivalenceKey"] == selected_class:
        v0_rows, v4_rows = int(row["multiplicity"]) - 1, 0
    else:
        v0_rows, v4_rows = int(row["multiplicity"]), 1
    worked_rows.append({"classKey": row["equivalenceKey"], "selectedClass": bool(row["selectedClass"]), "multiplicity": int(row["multiplicity"]), "V0PairRows": v0_rows, "V0WeightEach": 1.0 / raw_alternatives, "V4PairRows": v4_rows, "V4WeightEach": 0.0 if v4_rows == 0 else 1.0 / len(non_selected_classes)})
worked_table = pd.DataFrame(worked_rows)
assert np.isclose((worked_table.V0PairRows * worked_table.V0WeightEach).sum(), 1.0)
assert np.isclose((worked_table.V4PairRows * worked_table.V4WeightEach).sum(), 1.0)
display(pd.DataFrame([{"decisionKey": WORKED_DECISION, "rawCandidates": len(worked), "physicalClasses": len(worked_multiplicity), "selectedClassMultiplicity": int(worked_multiplicity.loc[worked_multiplicity.selectedClass, "multiplicity"].iloc[0])}]))
display(worked_table)


## 7. V0/V4 pair, shared scaler, estimator

**왜 실행하는가:** 두 모델에서 representation만 바뀌도록 scaler·feature·C·estimator를 공유한다.

**입력:** base 26게임의 모든 raw candidate.

**계산:** scaler는 base raw candidate에 한 번 fit한다. V0/V4 pair를 같은 scaled vectors로 만들고 symmetric binary sample로 학습한다.

**읽는 법:** `n_samples_seen_`는 base candidate 수와 같고, C는 base 전체 decision 수로 계산된다.

**보류 조건:** validation row가 scaler/pair/weight/fit에 들어가거나 convergence가 실패하면 중단한다.


In [ ]:
def fit_shared_scaler(train_frame: pd.DataFrame, validation_frame: pd.DataFrame) -> tuple[StandardScaler, dict[int, np.ndarray], dict[int, np.ndarray]]:
    train_values = np.vstack(train_frame["featureVector"])
    validation_values = np.vstack(validation_frame["featureVector"])
    scaler = StandardScaler(with_mean=True, with_std=True)
    train_scaled = scaler.fit_transform(train_values)
    validation_scaled = scaler.transform(validation_values)
    assert int(scaler.n_samples_seen_) == len(train_frame)
    train_map = {int(index): train_scaled[position] for position, index in enumerate(train_frame["candidateIndex"])}
    validation_map = {int(index): validation_scaled[position] for position, index in enumerate(validation_frame["candidateIndex"])}
    return scaler, train_map, validation_map


def build_pairs(frame: pd.DataFrame, scaled_by_index: dict[int, np.ndarray], variant: str) -> tuple[pd.DataFrame, list[str]]:
    assert variant in VARIANT_NAMES
    records, pairless = [], []
    for decision_key in sorted(frame["decisionKey"].unique(), key=codepoint_key):
        decision = frame.loc[frame["decisionKey"] == decision_key]
        selected = decision.loc[decision["selectedBySource"]].iloc[0]
        alternatives = decision.loc[~decision["selectedBySource"]].sort_values("placementKey", key=lambda column: column.map(codepoint_key))
        raw_count = len(alternatives)
        if variant == "V0":
            pair_rows = [(row, 1.0 / raw_count) for _, row in alternatives.iterrows()]
        else:
            physical = alternatives.loc[alternatives["equivalenceKey"] != selected["equivalenceKey"]]
            class_keys = sorted(physical["equivalenceKey"].unique(), key=codepoint_key)
            pair_rows = [(physical.loc[physical["equivalenceKey"] == class_key].iloc[0], 1.0 / len(class_keys)) for class_key in class_keys]
        if not pair_rows:
            pairless.append(decision_key)
            continue
        for alternative, pair_weight in pair_rows:
            records.append({"decisionKey": decision_key, "pairWeight": float(pair_weight), "difference": scaled_by_index[int(selected["candidateIndex"])] - scaled_by_index[int(alternative["candidateIndex"])], "alternativeClassKey": alternative["equivalenceKey"]})
    return pd.DataFrame(records), pairless


SCALER, TRAIN_SCALED, VALIDATION_SCALED = fit_shared_scaler(train_candidates, validation_candidates)
TRAIN_DECISION_COUNT = train_candidates["decisionKey"].nunique()
PAIR_FRAMES, PAIRLESS = {}, {}
for variant in VARIANT_NAMES:
    PAIR_FRAMES[variant], PAIRLESS[variant] = build_pairs(train_candidates, TRAIN_SCALED, variant)
display(pd.DataFrame([{"variant": variant, "pairCount": len(PAIR_FRAMES[variant]), "pairlessDecisionCount": len(PAIRLESS[variant]), "weightTotal": float(PAIR_FRAMES[variant].pairWeight.sum())} for variant in VARIANT_NAMES]))


In [ ]:
def create_symmetric_samples(pairs: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray, dict[str, Any]]:
    differences = np.vstack(pairs["difference"])
    weights = pairs["pairWeight"].to_numpy(dtype=np.float64)
    x_values = np.empty((len(pairs) * 2, differences.shape[1]), dtype=np.float64)
    labels = np.empty(len(pairs) * 2, dtype=np.int64)
    sample_weights = np.empty(len(pairs) * 2, dtype=np.float64)
    x_values[0::2], x_values[1::2] = differences, -differences
    labels[0::2], labels[1::2] = 1, 0
    sample_weights[0::2], sample_weights[1::2] = weights / 2.0, weights / 2.0
    decision_keys = np.repeat(pairs["decisionKey"].to_numpy(), 2)
    actual = pd.DataFrame({"decisionKey": decision_keys, "weight": sample_weights}).groupby("decisionKey", sort=False)["weight"].sum()
    expected = pairs.groupby("decisionKey", sort=False)["pairWeight"].sum()
    assert np.allclose(actual.loc[expected.index], expected, rtol=0.0, atol=1e-12)
    return x_values, labels, sample_weights, {"symmetricSampleCount": int(len(x_values)), "sampleWeightTotal": float(sample_weights.sum()), "minimumDecisionWeight": float(actual.min()), "maximumDecisionWeight": float(actual.max())}


def fit_pairwise_model(pairs: pd.DataFrame, full_train_decision_count: int) -> tuple[LogisticRegression, dict[str, Any]]:
    x_values, labels, sample_weights, weight_summary = create_symmetric_samples(pairs)
    c_value = 1.0 / (full_train_decision_count * MODEL_CONFIG["l2"])
    model = LogisticRegression(C=c_value, solver=MODEL_CONFIG["solver"], fit_intercept=MODEL_CONFIG["fitIntercept"], class_weight=MODEL_CONFIG["classWeight"], tol=MODEL_CONFIG["tolerance"], max_iter=MODEL_CONFIG["maxIterations"])
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        model.fit(x_values, labels, sample_weight=sample_weights)
    convergence = [str(item.message) for item in caught if issubclass(item.category, ConvergenceWarning)]
    unexpected = [f"{item.category.__name__}:{item.message}" for item in caught if not issubclass(item.category, ConvergenceWarning)]
    assert not convergence, convergence
    assert int(model.n_iter_[0]) < MODEL_CONFIG["maxIterations"]
    assert np.isfinite(model.coef_).all()
    return model, {"c": float(c_value), "iterations": int(model.n_iter_[0]), "convergenceWarnings": convergence, "unexpectedWarnings": unexpected, **weight_summary}


MODELS, TRAIN_DIAGNOSTICS = {}, {}
for variant in VARIANT_NAMES:
    MODELS[variant], TRAIN_DIAGNOSTICS[variant] = fit_pairwise_model(PAIR_FRAMES[variant], TRAIN_DECISION_COUNT)
assert TRAIN_DECISION_COUNT == BASE_MANIFEST["acceptedPlacementLogCount"] == 1158
display(pd.DataFrame([{"variant": variant, **TRAIN_DIAGNOSTICS[variant]} for variant in VARIANT_NAMES]))


## 8. 신규 game held-out 평가

**왜 실행하는가:** 학습에 없던 신규 game에서 raw compatibility metric과 primary physical-class metric을 함께 본다.

**입력:** validation candidates와 V0/V4 coefficient.

**계산:** canonical score ordering으로 selected rank를 구한다. Choice-required는 physical class가 둘 이상인 decision만 포함한다.

**읽는 법:** primary는 choice-required physical-class Top-1/Top-3/MRR/mean rank다. Raw metric은 진단이다.

**보류 조건:** score가 finite가 아니거나 두 variant의 decision 집합이 다르면 중단한다.


In [ ]:
def score_rows(rows: list[dict[str, Any]], game_id: str, decision_key: str, evaluation_unit: str) -> dict[str, Any]:
    assert rows and sum(row["selected"] for row in rows) == 1
    assert all(np.isfinite(row["score"]) for row in rows)
    ranked = sorted(rows, key=lambda row: (-row["score"], codepoint_key(row["key"])))
    selected = next(row for row in rows if row["selected"])
    selected_rank = next(index for index, row in enumerate(ranked, 1) if row["key"] == selected["key"])
    top_score = ranked[0]["score"]
    return {"gameId": game_id, "decisionKey": decision_key, "evaluationUnit": evaluation_unit, "candidateCount": len(rows), "choiceRequired": len(rows) > 1, "selectedRank": selected_rank, "selectedAtTopScore": bool(selected["score"] == top_score), "topScoreTieCount": int(sum(row["score"] == top_score for row in rows)), "selectedScore": float(selected["score"])}


def score_raw_candidates(frame: pd.DataFrame, scaled: dict[int, np.ndarray], coefficients: np.ndarray) -> list[dict[str, Any]]:
    predictions = []
    for decision_key in sorted(frame["decisionKey"].unique(), key=codepoint_key):
        group = frame.loc[frame["decisionKey"] == decision_key]
        rows = [{"key": row["placementKey"], "selected": bool(row["selectedBySource"]), "score": float(scaled[int(row["candidateIndex"])] @ coefficients)} for _, row in group.iterrows()]
        predictions.append(score_rows(rows, str(group["gameId"].iloc[0]), decision_key, "rawCandidate"))
    return predictions


def score_physical_classes(frame: pd.DataFrame, scaled: dict[int, np.ndarray], coefficients: np.ndarray) -> list[dict[str, Any]]:
    predictions = []
    for decision_key in sorted(frame["decisionKey"].unique(), key=codepoint_key):
        decision = frame.loc[frame["decisionKey"] == decision_key]
        rows = []
        for class_key in sorted(decision["equivalenceKey"].unique(), key=codepoint_key):
            members = decision.loc[decision["equivalenceKey"] == class_key].sort_values("placementKey", key=lambda column: column.map(codepoint_key))
            representative = members.iloc[0]
            rows.append({"key": class_key, "selected": bool(members["selectedBySource"].any()), "score": float(scaled[int(representative["candidateIndex"])] @ coefficients)})
        predictions.append(score_rows(rows, str(decision["gameId"].iloc[0]), decision_key, "physicalClass"))
    return predictions


PREDICTIONS = {}
for variant in VARIANT_NAMES:
    coefficients = MODELS[variant].coef_[0]
    PREDICTIONS[variant] = {"rawCandidate": score_raw_candidates(validation_candidates, VALIDATION_SCALED, coefficients), "physicalClass": score_physical_classes(validation_candidates, VALIDATION_SCALED, coefficients)}
assert {row["decisionKey"] for row in PREDICTIONS["V0"]["physicalClass"]} == {row["decisionKey"] for row in PREDICTIONS["V4"]["physicalClass"]}


In [ ]:
def summarize_predictions(predictions: list[dict[str, Any]]) -> dict[str, Any]:
    frame = pd.DataFrame(predictions)
    assert not frame.empty
    ranks = frame["selectedRank"].to_numpy(dtype=np.int64)
    return {"decisionCount": int(len(frame)), "top1Count": int((ranks == 1).sum()), "top3Count": int((ranks <= 3).sum()), "top1Accuracy": float(np.mean(ranks == 1)), "top3Accuracy": float(np.mean(ranks <= 3)), "mrr": float(np.mean(1.0 / ranks)), "meanSelectedRank": float(np.mean(ranks)), "tieAwareTop1Count": int(frame["selectedAtTopScore"].sum()), "topScoreTieDecisionCount": int((frame["topScoreTieCount"] > 1).sum())}


def evaluation_summary(predictions: list[dict[str, Any]]) -> dict[str, Any]:
    choice_required = [row for row in predictions if row["choiceRequired"]]
    per_game = []
    for game_id in sorted({row["gameId"] for row in predictions}, key=codepoint_key):
        game_rows = [row for row in predictions if row["gameId"] == game_id]
        game_choice = [row for row in game_rows if row["choiceRequired"]]
        per_game.append({"gameId": game_id, "all": summarize_predictions(game_rows), "choiceRequired": summarize_predictions(game_choice)})
    worst = min(per_game, key=lambda row: (row["choiceRequired"]["mrr"], codepoint_key(row["gameId"])))
    return {"all": summarize_predictions(predictions), "choiceRequired": summarize_predictions(choice_required), "perGame": per_game, "worstGame": {"gameId": worst["gameId"], **worst["choiceRequired"]}}


EVALUATION = {variant: {unit: evaluation_summary(PREDICTIONS[variant][unit]) for unit in ["physicalClass", "rawCandidate"]} for variant in VARIANT_NAMES}
primary_table = pd.DataFrame([{"variant": variant, **EVALUATION[variant]["physicalClass"]["choiceRequired"]} for variant in VARIANT_NAMES])
display(primary_table)


## 9. 진단과 결정적 smoke report

**왜 실행하는가:** metric만으로 숨겨지는 pairless, duplicate, tie, rank delta와 source 불변성을 함께 전달한다.

**입력:** 두 model의 held-out prediction, dataset/source checksum.

**계산:** game별 worst case, coefficient/score/rank 변화와 deterministic JSON payload를 만든다.

**읽는 법:** smoke report의 status는 `awaiting_full_temporal_evaluation`이고 policy/canonical eligibility는 false다.

**보류 조건:** leakage, convergence, finite score, source checksum 중 하나라도 깨지면 report를 성공으로 해석하지 않는다.


In [ ]:
def rank_map(predictions: list[dict[str, Any]]) -> dict[str, int]:
    return {row["decisionKey"]: int(row["selectedRank"]) for row in predictions}


def selected_feature_duplicate_count(frame: pd.DataFrame) -> int:
    count = 0
    for decision_key in frame["decisionKey"].unique():
        group = frame.loc[frame["decisionKey"] == decision_key]
        selected = group.loc[group["selectedBySource"], "featureVector"].iloc[0]
        alternatives = group.loc[~group["selectedBySource"], "featureVector"]
        count += int(any(np.array_equal(selected, candidate) for candidate in alternatives))
    return count


raw_v0, raw_v4 = rank_map(PREDICTIONS["V0"]["rawCandidate"]), rank_map(PREDICTIONS["V4"]["rawCandidate"])
class_v0, class_v4 = rank_map(PREDICTIONS["V0"]["physicalClass"]), rank_map(PREDICTIONS["V4"]["physicalClass"])
coefficient_delta = MODELS["V4"].coef_[0] - MODELS["V0"].coef_[0]
score_delta = []
for decision_key in sorted(raw_v0, key=codepoint_key):
    v0_row = next(row for row in PREDICTIONS["V0"]["rawCandidate"] if row["decisionKey"] == decision_key)
    v4_row = next(row for row in PREDICTIONS["V4"]["rawCandidate"] if row["decisionKey"] == decision_key)
    score_delta.append(abs(v4_row["selectedScore"] - v0_row["selectedScore"]))
SOURCE_SHA256_AFTER = sha256_file(SOURCE_LOG)
BASE_SHA256_AFTER = {name: sha256_file(BASE_DATASET / name) for name in DATASET_FILES}
CUMULATIVE_SHA256_AFTER = {name: sha256_file(CUMULATIVE_DATASET / name) for name in DATASET_FILES}
assert SOURCE_SHA256_AFTER == SOURCE_SHA256_BEFORE, "SOURCE_LOG_MUTATED"
assert BASE_SHA256_AFTER == BASE_SHA256, "BASE_DATASET_MUTATED"
assert CUMULATIVE_SHA256_AFTER == CUMULATIVE_SHA256, "CUMULATIVE_DATASET_MUTATED"
STATUS = "needs_policy_decision" if FULL_COHORT_COVERED else "awaiting_full_temporal_evaluation"
POLICY_DECISION_ELIGIBLE = bool(FULL_COHORT_COVERED)
CANONICAL_RESULT_ELIGIBLE = False


In [ ]:
REPORT = {
    "experimentId": EXPERIMENT_ID, "status": STATUS, "runMode": RUN_MODE,
    "fullCohortCovered": FULL_COHORT_COVERED, "policyDecisionEligible": POLICY_DECISION_ELIGIBLE,
    "canonicalResultEligible": CANONICAL_RESULT_ELIGIBLE, "productionIntegration": False,
    "datasetChecksums": {"base": BASE_SHA256, "cumulative": CUMULATIVE_SHA256},
    "datasetChecksumsAfter": {"base": BASE_SHA256_AFTER, "cumulative": CUMULATIVE_SHA256_AFTER},
    "sourceChecksumBefore": SOURCE_SHA256_BEFORE, "sourceChecksumAfter": SOURCE_SHA256_AFTER,
    "cohorts": {"baseGameCount": len(BASE_GAME_IDS), "cumulativeGameCount": len(CUMULATIVE_GAME_IDS), "newGameCount": len(NEW_GAME_IDS), "evaluatedNewGameCount": len(VALIDATION_GAME_IDS), "baseGameIdDigest": hashlib.sha256("\n".join(sorted(BASE_GAME_IDS, key=codepoint_key)).encode()).hexdigest(), "newGameIdDigest": hashlib.sha256("\n".join(NEW_GAME_IDS).encode()).hexdigest(), "evaluatedGameIds": VALIDATION_GAME_IDS},
    "featureContract": {"featureNames": FEATURE_NAMES, "forbiddenLeakageInputs": sorted(FORBIDDEN_FEATURES), "scalerFitScope": "base-26-game raw training candidates only", "scalerFitCandidateCount": int(SCALER.n_samples_seen_)},
    "estimatorConfig": MODEL_CONFIG,
    "training": {variant: {"pairCount": len(PAIR_FRAMES[variant]), "pairlessDecisionCount": len(PAIRLESS[variant]), **TRAIN_DIAGNOSTICS[variant]} for variant in VARIANT_NAMES},
    "evaluation": EVALUATION,
    "diagnostics": {"gameIdLeakageCount": GAME_ID_LEAKAGE_COUNT, "selectedFeatureDuplicateDecisionCount": selected_feature_duplicate_count(validation_candidates), "rawCandidateRankChangeCount": int(sum(raw_v0[key] != raw_v4[key] for key in raw_v0)), "physicalClassRankChangeCount": int(sum(class_v0[key] != class_v4[key] for key in class_v0)), "coefficientL2Difference": float(np.linalg.norm(coefficient_delta)), "maximumAbsoluteCoefficientDifference": float(np.max(np.abs(coefficient_delta))), "maximumAbsoluteSelectedScoreDifference": float(max(score_delta))},
}
report_text = json.dumps(REPORT, ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False) + "\n"


def same_inode(left: Path, right: Path) -> bool:
    left_stat, right_stat = left.stat(), right.stat()
    return (left_stat.st_dev, left_stat.st_ino) == (right_stat.st_dev, right_stat.st_ino)


def atomic_write_text(output_path: Path, text: str, protected_sources: list[Path]) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_real = output_path.resolve(strict=False)
    for source_path in protected_sources:
        assert output_real != source_path.resolve(strict=True), "REPORT_SOURCE_COLLISION"
        if os.path.lexists(output_path):
            assert not output_path.is_symlink(), "REPORT_OUTPUT_SYMLINK_NOT_ALLOWED"
            assert not same_inode(output_path, source_path), "REPORT_OUTPUT_INODE_COLLISION"
    descriptor, staging_name = tempfile.mkstemp(prefix=f".{output_path.name}.", suffix=".tmp", dir=output_path.parent)
    staging_path = Path(staging_name)
    descriptor_stat = os.fstat(descriptor)
    try:
        with os.fdopen(descriptor, "w", encoding="utf-8", newline="\n") as handle:
            handle.write(text)
            handle.flush()
            os.fsync(handle.fileno())
        staging_stat = staging_path.lstat()
        assert not staging_path.is_symlink()
        assert (staging_stat.st_dev, staging_stat.st_ino) == (descriptor_stat.st_dev, descriptor_stat.st_ino)
        assert all(not same_inode(staging_path, source) for source in protected_sources)
        os.replace(staging_path, output_path)
    finally:
        if staging_path.exists():
            staging_path.unlink()


REPORT_PATH = resolve_config_path(os.environ.get("PLACEMENT_TEMPORAL_REPORT", "/private/tmp/placement-temporal-holdout-smoke-report.json"), ROOT)
PROTECTED_SOURCES = [SOURCE_LOG, *[BASE_DATASET / name for name in DATASET_FILES], *[CUMULATIVE_DATASET / name for name in DATASET_FILES]]
atomic_write_text(REPORT_PATH, report_text, PROTECTED_SOURCES)
REPORT_SHA256 = hashlib.sha256(report_text.encode("utf-8")).hexdigest()
assert sha256_file(REPORT_PATH) == REPORT_SHA256
display(pd.DataFrame([{"status": STATUS, "runMode": RUN_MODE, "reportSha256": REPORT_SHA256, "sourceUnchanged": SOURCE_SHA256_AFTER == SOURCE_SHA256_BEFORE}]))


## 10. 결과를 읽는 법과 user full-run gate

관측: smoke는 base 26게임 전체로 학습하지만 정렬상 첫 2개 신규 game만 평가한다. 따라서 pipeline, leakage, finite score와 report contract만 확인한다.

해석: Smoke 결과로 V0 또는 V4를 채택하지 않는다. `awaiting_full_temporal_evaluation`은 구현이 full cohort 실행을 기다린다는 뜻이다.

결정: full 실행은 사용자가 명시적으로 수행한다. Full report도 자동 채택하지 않고 `needs_policy_decision`으로 기술 리드에게 넘긴다.

사용자 full-run 명령:

```bash
PLACEMENT_TEMPORAL_SOURCE_LOG=../Tetris_multi/data/logs/game-events.jsonl \
PLACEMENT_TEMPORAL_CUMULATIVE_DATASET=data/datasets/human-behavior-v2-temporal-cumulative-75g-20260718 \
PLACEMENT_TEMPORAL_MAX_VALIDATION_GAMES=49 \
PLACEMENT_TEMPORAL_REPORT=/private/tmp/placement-temporal-holdout-full-report.json \
conda run -n tetris-ml python -m jupyter nbconvert --to notebook --execute \
  scripts/ml/notebooks/placement_temporal_holdout_v0_v4_experiment.ipynb \
  --output placement-temporal-holdout-full-executed.ipynb --output-dir /private/tmp \
  --ExecutePreprocessor.timeout=1200
```

아직 말할 수 없는 것: 49게임 전체의 V0/V4 우열, gameplay 승률, production 적합성, representation freeze 여부.
